In [1]:
# ===========================
# IMPORTS
# ===========================
import requests
import pandas as pd


In [2]:
# ===========================
# FETCH ALL GAMEWEEK DATA
# ===========================

all_gw_data = []

for gw in range(1, 39):   # up to GW38
    url = f"https://fantasy.premierleague.com/api/event/{gw}/live/"
    data = requests.get(url).json()
    
    for p in data['elements']:
        # Add gameweek number to each player
        p['gw'] = gw
        all_gw_data.append(p)

# Convert to DataFrame
stats_df = pd.DataFrame(all_gw_data)

# Save to CSV (optional)
stats_df.to_csv("fpl_2025_26.csv", index=False)

print("Raw FPL data fetched. Shape:", stats_df.shape)
print(stats_df.head())


Raw FPL data fetched. Shape: (12619, 5)
   id                                              stats  \
0   1  {'minutes': 90, 'goals_scored': 0, 'assists': ...   
1   2  {'minutes': 0, 'goals_scored': 0, 'assists': 0...   
2   3  {'minutes': 0, 'goals_scored': 0, 'assists': 0...   
3   4  {'minutes': 0, 'goals_scored': 0, 'assists': 0...   
4   5  {'minutes': 90, 'goals_scored': 0, 'assists': ...   

                                             explain  modified  gw  
0  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
1  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
2  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
3  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
4  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  


In [32]:
# ===========================
# EXPAND STATS COLUMN
# ===========================

# The 'stats' column contains a dict of all player stats for the GW
stats_expanded = stats_df['stats'].apply(pd.Series)

# Add back player id and GW
stats_expanded['id'] = stats_df['id']
stats_expanded['gw'] = stats_df['gw']

# Your clean expanded DataFrame
df_clean = stats_expanded

print("Expanded FPL stats dataframe. Shape:", df_clean.shape)
print(df_clean.head())
print(df_clean.columns)


Expanded FPL stats dataframe. Shape: (8818, 30)
   minutes  goals_scored  assists  clean_sheets  goals_conceded  own_goals  \
0       90             0        0             1               0          0   
1        0             0        0             0               0          0   
2        0             0        0             0               0          0   
3        0             0        0             0               0          0   
4       90             0        0             1               0          0   

   penalties_saved  penalties_missed  yellow_cards  red_cards  ...  \
0                0                 0             1          0  ...   
1                0                 0             0          0  ...   
2                0                 0             0          0  ...   
3                0                 0             0          0  ...   
4                0                 0             0          0  ...   

   defensive_contribution  starts  expected_goals expected_ass

In [33]:
# ===========================
# FETCH ADDITIONAL DATA FROM FPL API
# ===========================

import requests
import pandas as pd

# 1️⃣ Bootstrap data (players and teams)
bootstrap_url = "https://fantasy.premierleague.com/api/bootstrap-static/"
bootstrap_data = requests.get(bootstrap_url).json()

# Player info
players_df = pd.DataFrame(bootstrap_data['elements'])
# Keep only relevant player-level columns
players_df = players_df[[
    'id', 'first_name', 'second_name', 'team', 'element_type',
    'now_cost', 'chance_of_playing_next_round', 'form',
    'influence', 'creativity', 'threat', 'ict_index'
]]

# Team info
teams_df = pd.DataFrame(bootstrap_data['teams'])
teams_df = teams_df[[
    'id', 'strength_overall_home', 'strength_overall_away',
    'strength_attack_home', 'strength_attack_away',
    'strength_defence_home', 'strength_defence_away',
    'form'
]]

# Fixtures info
fixtures_url = "https://fantasy.premierleague.com/api/fixtures/"
fixtures_df = pd.DataFrame(requests.get(fixtures_url).json())
# Keep only relevant fixture columns
fixtures_df = fixtures_df[[
    'id', 'event', 'team_h', 'team_a', 'kickoff_time', 
    'minutes', 'team_h_difficulty', 'team_a_difficulty'
]]

# Print shapes and columns to verify
print("Players shape:", players_df.shape)
print("Teams shape:", teams_df.shape)
print("Fixtures shape:", fixtures_df.shape)
print ("players cols:" , players_df.columns)
print ("teams cols:" , teams_df.columns)
print ("fixtures cols:" , fixtures_df.columns)


Players shape: (755, 12)
Teams shape: (20, 8)
Fixtures shape: (380, 8)
players cols: Index(['id', 'first_name', 'second_name', 'team', 'element_type', 'now_cost',
       'chance_of_playing_next_round', 'form', 'influence', 'creativity',
       'threat', 'ict_index'],
      dtype='object')
teams cols: Index(['id', 'strength_overall_home', 'strength_overall_away',
       'strength_attack_home', 'strength_attack_away', 'strength_defence_home',
       'strength_defence_away', 'form'],
      dtype='object')
fixtures cols: Index(['id', 'event', 'team_h', 'team_a', 'kickoff_time', 'minutes',
       'team_h_difficulty', 'team_a_difficulty'],
      dtype='object')


In [34]:
# ===========================
# MERGE PLAYER, TEAM, AND FIXTURE INFO (FIXED)
# ===========================

# 1️⃣ Merge player info to get names, team, cost, etc.
df_with_player = df_clean.merge(
    players_df,
    on='id',
    how='left'
)

print("Columns after merging player info:", df_with_player.columns)

# 2️⃣ Merge home fixtures
df_home = df_with_player.merge(
    fixtures_df,
    left_on=['gw', 'team'],
    right_on=['event', 'team_h'],
    how='left'
)
df_home['is_home'] = 1
df_home['opponent_team_id'] = df_home['team_a']
df_home['opponent_difficulty'] = df_home['team_h_difficulty']

# 3️⃣ Merge away fixtures
df_away = df_with_player.merge(
    fixtures_df,
    left_on=['gw', 'team'],
    right_on=['event', 'team_a'],
    how='left'
)
df_away['is_home'] = 0
df_away['opponent_team_id'] = df_away['team_h']
df_away['opponent_difficulty'] = df_away['team_a_difficulty']

# 4️⃣ Concatenate home and away players
# Keep only columns present in both dataframes
cols_to_keep = list(set(df_home.columns).intersection(set(df_away.columns)))
df_merged = pd.concat([df_home[cols_to_keep], df_away[cols_to_keep]], ignore_index=True)

# 5️⃣ Merge team info (optional)
df_merged = df_merged.merge(
    teams_df,
    left_on='team',
    right_on='id',
    suffixes=('', '_team'),
    how='left'
)

# Safely drop 'id_team' if it exists
if 'id_team' in df_merged.columns:
    df_merged.drop(columns=['id_team'], inplace=True)

# 6️⃣ Check final dataset
print("Merged dataset shape:", df_merged.shape)
print(df_merged[['id', 'gw', 'team', 'opponent_team_id', 'is_home', 'opponent_difficulty']].head())
print(df_merged.columns)


Columns after merging player info: Index(['minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
       'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
       'red_cards', 'saves', 'bonus', 'bps', 'influence_x', 'creativity_x',
       'threat_x', 'ict_index_x', 'clearances_blocks_interceptions',
       'recoveries', 'tackles', 'defensive_contribution', 'starts',
       'expected_goals', 'expected_assists', 'expected_goal_involvements',
       'expected_goals_conceded', 'total_points', 'in_dreamteam', 'id', 'gw',
       'first_name', 'second_name', 'team', 'element_type', 'now_cost',
       'chance_of_playing_next_round', 'form', 'influence_y', 'creativity_y',
       'threat_y', 'ict_index_y'],
      dtype='object')
Merged dataset shape: (17636, 60)
   id  gw  team  opponent_team_id  is_home  opponent_difficulty
0   1   1     1               NaN        1                  NaN
1   1   1     1               NaN        1                  NaN
2   1   1   